In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


In [ ]:
!mamba create -y -n vep_env -c bioconda -c conda-forge ensembl-vep python=3.10


Looking for: ['ensembl-vep', 'python=3.10']

bioconda/linux-64                                           Using cache
bioconda/noarch                                             Using cache
conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache
Transaction

  Prefix: /usr/local/envs/vep_env

  Updating specs:

   - ensembl-vep
   - python=3.10


  Package                                           Version  Build                  Channel           Size
────────────────────────────────────────────────────────────────────────────────────────────────────────────
  Install:
────────────────────────────────────────────────────────────────────────────────────────────────────────────

  + libgomp                                          15.2.0  he0feb66_18            conda-forge      603kB
  + mpi                                                 1.0  openmpi                conda-forge        4kB
  + ld_impl_

In [ ]:
!conda run -n vep_env vep --help | head -20

#----------------------------------#
# ENSEMBL VARIANT EFFECT PREDICTOR #
#----------------------------------#

Versions:
  ensembl              : 115.266b84d
  ensembl-compara      : 115.ae48a7a
  ensembl-funcgen      : 115.57f7061
  ensembl-io           : 115.25061d3
  ensembl-variation    : 115.b7c2637
  ensembl-vep          : 115.2

Help: dev@ensembl.org , helpdesk@ensembl.org
Twitter: @ensembl

http://www.ensembl.org/info/docs/tools/vep/script/index.html

Usage:
./vep [--cache|--offline|--database] [arguments]



In [ ]:
!conda run -n vep_env vep_install -a cf -s homo_sapiens -y GRCh38 -c ~/.vep

 - getting list of available cache files
 - downloading https://ftp.ensembl.org/pub/release-115/variation/indexed_vep_cache/homo_sapiens_vep_115_GRCh38.tar.gz
 - unpacking homo_sapiens_vep_115_GRCh38.tar.gz
 - converting cache, this may take some time but will allow VEP to look up variants and frequency data much faster
 - use CTRL-C to cancel if you do not wish to convert this cache now (you may run convert_cache.pl later)
2026-02-24 11:42:52 - Processing homo_sapiens
2026-02-24 11:42:52 - Processing version 115_GRCh38
2026-02-24 11:42:52 - No unprocessed types remaining, skipping
2026-02-24 11:42:52 - All done!
 - downloading Homo_sapiens.GRCh38.dna.toplevel.fa.gz
 - downloading Homo_sapiens.GRCh38.dna.toplevel.fa.gz.fai
 - downloading Homo_sapiens.GRCh38.dna.toplevel.fa.gz.gzi

The FASTA file should be automatically detected by the VEP when using --cache or --offline.
If it is not, use "--fasta /root/.vep/homo_sapiens/115_GRCh38/Homo_sapiens.GRCh38.dna.toplevel.fa.gz"


All done

cu

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Galaxy28-[VarScan somatic on dataset 6, 25, and 27].vcf to Galaxy28-[VarScan somatic on dataset 6, 25, and 27] (1).vcf


In [ ]:
import subprocess

input_vcf = "Galaxy28-[VarScan somatic on dataset 6, 25, and 27] (1).vcf"
output_vcf = "annotated.vep.vcf"

cmd = f'''
conda run -n vep_env vep \
 -i "{input_vcf}" \
 -o {output_vcf} \
 --cache \
 --dir_cache ~/.vep \
 --assembly GRCh38 \
 --vcf \
 --sift b \
 --polyphen b \
 --force_overwrite
'''

subprocess.run(cmd, shell=True, check=True)

print("✅ VEP annotation finished")

✅ VEP annotation finished


In [ ]:
!ls -lh annotated.vep.vcf

-rw-r--r-- 1 root root 405M Feb 24 12:33 annotated.vep.vcf


In [ ]:
from google.colab import files
files.download("annotated.vep.vcf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import csv

input_vcf = "annotated.vep.vcf"
output_csv = "sift_polyphen_clean.csv"

csq_index = None
sift_idx = None
polyphen_idx = None

results = []

with open(input_vcf) as f:
    for line in f:
        # 🔹 Find CSQ header definition
        if line.startswith("##INFO=<ID=CSQ"):
            desc = line.split("Format: ")[1].strip('"\n>') # Revert to include '>' in stripping
            fields = desc.split("|")

            sift_idx = fields.index("SIFT")
            polyphen_idx = fields.index("PolyPhen")
            continue

        if line.startswith("#"):
            continue

        cols = line.strip().split("\t")
        info_field = cols[7]

        if "CSQ=" not in info_field:
            continue

        csq_data = info_field.split("CSQ=")[1].split(";")[0]
        first_transcript = csq_data.split(",")[0]
        parts = first_transcript.split("|")

        sift_val = parts[sift_idx] if sift_idx < len(parts) else "NA"
        polyphen_val = parts[polyphen_idx] if polyphen_idx < len(parts) else "NA"

        results.append({
            "CHROM": cols[0],
            "POS": cols[1],
            "REF": cols[3],
            "ALT": cols[4],
            "SIFT": sift_val,
            "PolyPhen": polyphen_val
        })

with open(output_csv, "w", newline="") as out:
    writer = csv.DictWriter(out, fieldnames=[
        "CHROM","POS","REF","ALT","SIFT","PolyPhen"
    ])
    writer.writeheader()
    writer.writerows(results)

print("✅ Clean extraction complete")

✅ Clean extraction complete


In [ ]:
from google.colab import files
files.download("sift_polyphen_clean.tsv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Galaxy118-[Join two Datasets on dataset 105 and 117] (1).tabular to Galaxy118-[Join two Datasets on dataset 105 and 117] (1).tabular


In [ ]:
import os
print(os.listdir())

['.config', 'sift_polyphen_clean.tsv', 'sample_data']


In [ ]:
from google.colab import files
files.upload()

Saving SRR19077358_SRR19077359_Clinvar_comepnedium_Final_analysis file.xlsx to SRR19077358_SRR19077359_Clinvar_comepnedium_Final_analysis file.xlsx
Buffered data was truncated after reaching the output size limit.

In [ ]:
import os
print(os.listdir())

['.config', 'SRR19077358_SRR19077359_Clinvar_comepnedium_Final_analysis file.xlsx', 'sift_polyphen_clean.tsv', 'sample_data']
